In [1]:
# load dungeon data

import json
import random
from pathlib import Path

data_path = Path("../datasets/dungeon_10k_4_8_3_5_mkr.jsonl")
with open(data_path) as f:
    data = [json.loads(line) for line in f]

# strip _id fields
for entry in data:
    if "_id" in entry:
        del entry["_id"]

# Split into train/eval sets
random.shuffle(data)
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
eval_data = data[split_idx:]

print(f"Train set: {len(train_data)} records")
print(f"Eval set: {len(eval_data)} records")

# print first data point
print(json.dumps(data[0], indent=2))

Train set: 8000 records
Eval set: 2000 records
{
  "door": 2,
  "key_color": "green",
  "corridor": [
    {
      "monsters": [
        "goblin"
      ],
      "door_no": 2,
      "blue_key": "spellbooks",
      "green_key": "gemstones",
      "red_key": "gold"
    },
    {
      "monsters": [
        "troll",
        "goblin"
      ],
      "door_no": 0,
      "red_key": "gemstones",
      "green_key": "artifacts",
      "blue_key": "spellbooks"
    },
    {
      "monsters": [
        "dragon",
        "wolf"
      ],
      "door_no": 1,
      "blue_key": "gold",
      "red_key": "gemstones",
      "green_key": "gemstones"
    },
    {
      "monsters": [
        "goblin",
        "dragon"
      ],
      "door_no": 3,
      "green_key": "gold",
      "blue_key": "diamonds",
      "red_key": "spellbooks"
    },
    {
      "monsters": [
        "orc",
        "troll"
      ],
      "door_no": 4,
      "green_key": "gold",
      "blue_key": "gold",
      "red_key": "gold"
    }
  ],
  

In [ ]:
from origami.pipeline import OrigamiPipeline, PipelineConfig
from origami.training import TableLogCallback, accuracy

config = PipelineConfig(
    d_model=196,
    n_heads=4,
    n_layers=6,
    d_ff=784,
    dropout=0.0,
    shuffle_keys=False,
    upscale_factor=1,
    batch_size=100,
    warmup_steps=1000,
    learning_rate=5e-4,
    eval_strategy="epoch",
    eval_epochs=5,
    eval_metrics={"acc": accuracy},
    eval_sample_size=100,
    target_key="treasure",
    use_grammar_constraints=True,
)

pipeline = OrigamiPipeline(config)
pipeline.fit(train_data, eval_data=eval_data, callbacks=[TableLogCallback(print_every=50)], epochs=80)


| step: 50 | epoch: 0 | lr: 2.50e-05 | batch_dt: 102ms | loss: 2.3503 |
| step: 100 | epoch: 1 | lr: 5.00e-05 | batch_dt: 124ms | loss: 1.3614 |
| step: 150 | epoch: 1 | lr: 7.50e-05 | batch_dt: 100ms | loss: 1.0744 |
| step: 200 | epoch: 2 | lr: 1.00e-04 | batch_dt: 120ms | loss: 0.9596 |
| step: 250 | epoch: 3 | lr: 1.25e-04 | batch_dt: 98ms | loss: 0.8531 |
| step: 300 | epoch: 3 | lr: 1.50e-04 | batch_dt: 110ms | loss: 0.8233 |
| step: 350 | epoch: 4 | lr: 1.75e-04 | batch_dt: 103ms | loss: 0.8243 |


In [ ]:
pipeline.save("dungeon_pipeline.pt")

In [ ]:
pipeline.evaluate(eval_data, metrics={"acc": accuracy})